In [1]:
# ============================================================
# CABITZA ET AL. COVID-19 MODEL REPRODUCTION
# Logistic Regression
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    GridSearchCV
)

from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score,
    sensitivity_score if False else recall_score,
    confusion_matrix,
    roc_auc_score,
    brier_score_loss
)

# Reproducibility
RANDOM_STATE = 42

print("Imports successful.")
print("Random state:", RANDOM_STATE)

SyntaxError: invalid syntax (1567405123.py, line 25)

In [3]:
# ============================================================
# CABITZA ET AL. COVID-19 MODEL REPRODUCTION
# Logistic Regression
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    GridSearchCV
)

from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score,
    recall_score,
    confusion_matrix,
    roc_auc_score,
    brier_score_loss
)

RANDOM_STATE = 42

print("Imports successful.")
print("Random state:", RANDOM_STATE)

Imports successful.
Random state: 42


In [4]:
# ============================================================
# LOAD DATASET
# ============================================================

DATA_PATH = Path("../data/raw/all_training.csv")

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print("Shape:", df.shape)

display(df.head())

Dataset loaded successfully.
Shape: (1736, 36)


,Unnamed: 0,Sex,Age,CA,CK,CREA,ALP,GGT,GLU,AST,...,MO,EO,BA,NET,LYT,MOT,EOT,BAT,Suspect,target
0,A00345_2020-03-25,1.0,82.0,2.09,NaN,1.150,95.0,40.0,78.0,26.0,...,9.5,2.9,0.5,6.40,1.20,0.8,0.3,0.0,1.0,0
1,A00791_2020-03-19,1.0,51.0,1.97,237.0,0.970,54.0,98.0,98.0,74.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,1
2,A00741_2020-03-04,1.0,58.0,2.11,NaN,1.000,80.0,147.0,106.0,41.0,...,7.3,0.3,0.1,5.45,0.75,0.5,0.0,0.0,1.0,0
3,A00605_2020-04-15,0.0,82.0,2.27,138.0,0.755,123.5,176.5,106.0,114.0,...,9.5,1.7,0.9,3.60,2.60,0.7,0.1,0.1,0.5,0
4,A00417_2020-02-24,1.0,79.0,2.07,73.0,1.810,62.0,36.5,96.0,28.0,...,10.0,8.5,0.5,0.40,0.50,0.1,0.1,0.0,1.0,0


In [5]:
# ============================================================
# RECONSTRUCT STUDY COHORTS
# ============================================================

id_col = "Unnamed: 0"

ids = df[id_col].astype(str).str.strip()

def assign_cohort(patient_id):
    if patient_id.startswith("A"):
        return "training"
    elif patient_id.startswith("PSMAY"):
        return "training"
    elif patient_id.isdigit() and 2001 <= int(patient_id) <= 2058:
        return "internal_external"
    elif patient_id.isdigit():
        return "external"
    else:
        return "unclassified"

df["cohort"] = ids.apply(assign_cohort)

print(df["cohort"].value_counts())
print("\nTotal:", len(df))

cohort
training             1624
internal_external      58
external               54
Name: count, dtype: int64

Total: 1736


In [6]:
# ============================================================
# DEFINE PREDICTORS AND OUTCOME
# ============================================================

target_col = "target"

predictors = [
    col for col in df.columns
    if col not in [id_col, target_col, "cohort"]
]

print("Number of predictors:", len(predictors))

for i, feature in enumerate(predictors, start=1):
    print(f"{i:2d}. {feature}")

Number of predictors: 34
 1. Sex
 2. Age
 3. CA
 4. CK
 5. CREA
 6. ALP
 7. GGT
 8. GLU
 9. AST
10. ALT
11. LDH
12. PCR
13. KAL
14. NAT
15. UREA
16. WBC
17. RBC
18. HGB
19. HCT
20. MCV
21. MCH
22. MCHC
23. PLT1
24. NE
25. LY
26. MO
27. EO
28. BA
29. NET
30. LYT
31. MOT
32. EOT
33. BAT
34. Suspect


In [7]:
# ============================================================
# DEVELOPMENT DATASET
# ============================================================

development_df = df[df["cohort"] == "training"].copy()

X = development_df[predictors].copy()
y = development_df[target_col].copy()

print("Development cohort:", development_df.shape)
print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nOutcome counts:")
print(y.value_counts().sort_index())

print("\nOutcome proportions:")
print(y.value_counts(normalize=True).sort_index().round(4))

Development cohort: (1624, 37)
X shape: (1624, 34)
y shape: (1624,)

Outcome counts:
target
0    838
1    786
Name: count, dtype: int64

Outcome proportions:
target
0    0.516
1    0.484
Name: proportion, dtype: float64


In [8]:
# ============================================================
# 80:20 STRATIFIED TRAIN / HOLD-OUT SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

print("Training set:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("\nHold-out test set:")
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

print("\nTraining outcome distribution:")
print(y_train.value_counts().sort_index())
print(y_train.value_counts(normalize=True).sort_index().round(4))

print("\nHold-out outcome distribution:")
print(y_test.value_counts().sort_index())
print(y_test.value_counts(normalize=True).sort_index().round(4))

Training set:
X_train: (1299, 34)
y_train: (1299,)

Hold-out test set:
X_test: (325, 34)
y_test: (325,)

Training outcome distribution:
target
0    670
1    629
Name: count, dtype: int64
target
0    0.5158
1    0.4842
Name: proportion, dtype: float64

Hold-out outcome distribution:
target
0    168
1    157
Name: count, dtype: int64
target
0    0.5169
1    0.4831
Name: proportion, dtype: float64


In [9]:
# ============================================================
# 5-FOLD STRATIFIED CROSS-VALIDATION
# ============================================================

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

print("Cross-validation configured.")
print("Number of folds:", cv.get_n_splits())

Cross-validation configured.
Number of folds: 5


In [10]:
# ============================================================
# LOGISTIC REGRESSION PIPELINE
# Cabitza-style methodological reproduction
# ============================================================

# Base logistic regression model
lr = LogisticRegression(
    max_iter=5000,
    random_state=RANDOM_STATE
)

# Logistic regression estimator used by RFE
rfe_estimator = LogisticRegression(
    max_iter=5000,
    random_state=RANDOM_STATE
)

pipeline = Pipeline([
    ("imputer", KNNImputer(n_neighbors=5)),
    ("scaler", StandardScaler()),
    ("rfe", RFE(
        estimator=rfe_estimator,
        step=1
    )),
    ("classifier", lr)
])

print("Pipeline created successfully.")

for name, step in pipeline.steps:
    print(f"{name}: {step}")

Pipeline created successfully.
imputer: KNNImputer()
scaler: StandardScaler()
rfe: RFE(estimator=LogisticRegression(max_iter=5000, random_state=42))
classifier: LogisticRegression(max_iter=5000, random_state=42)


In [11]:
# ============================================================
# HYPERPARAMETER SEARCH SPACE
# ============================================================
# NOTE:
# The original publication states that hyperparameters were
# optimized by grid search, but the exact LR grid from
# Supplementary Table 1 is unavailable.
#
# Therefore, this reproduction uses a prespecified grid.

param_grid = {
    # Number of predictors retained by RFE
    "rfe__n_features_to_select": [
        5,
        10,
        15,
        20,
        25,
        30,
        34
    ],

    # Logistic regression regularization strength
    "classifier__C": [
        0.01,
        0.1,
        1,
        10,
        100
    ],

    # Regularization type
    "classifier__penalty": [
        "l1",
        "l2"
    ],

    # Solver compatible with both L1 and L2
    "classifier__solver": [
        "liblinear"
    ]
}

n_combinations = (
    len(param_grid["rfe__n_features_to_select"])
    * len(param_grid["classifier__C"])
    * len(param_grid["classifier__penalty"])
)

print("Hyperparameter grid created.")
print("Parameter combinations:", n_combinations)
print("CV folds:", cv.get_n_splits())
print("Total CV fits:", n_combinations * cv.get_n_splits())

Hyperparameter grid created.
Parameter combinations: 70
CV folds: 5
Total CV fits: 350


In [12]:
# ============================================================
# GRID SEARCH
# ============================================================

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)

print("GridSearchCV configured successfully.")
print("Optimization metric: ROC-AUC")

GridSearchCV configured successfully.
Optimization metric: ROC-AUC


In [13]:
# ============================================================
# MODEL TRAINING + HYPERPARAMETER OPTIMIZATION
# ============================================================

grid_search.fit(X_train, y_train)

print("\nGrid search completed.")

print("\nBest cross-validation ROC-AUC:")
print(round(grid_search.best_score_, 4))

print("\nBest hyperparameters:")
for parameter, value in grid_search.best_params_.items():
    print(f"{parameter}: {value}")

Fitting 5 folds for each of 70 candidates, totalling 350 fits

Grid search completed.

Best cross-validation ROC-AUC:
0.8702

Best hyperparameters:
classifier__C: 0.1
classifier__penalty: l1
classifier__solver: liblinear
rfe__n_features_to_select: 34


c:\Users\bl\Documents\ML-Projects\clinical-ml\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\bl\Documents\ML-Projects\clinical-ml\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1407: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


In [14]:
# ============================================================
# TOP CROSS-VALIDATION RESULTS
# ============================================================

cv_results = pd.DataFrame(grid_search.cv_results_)

top_results = (
    cv_results[
        [
            "param_rfe__n_features_to_select",
            "param_classifier__C",
            "param_classifier__penalty",
            "mean_test_score",
            "std_test_score",
            "mean_train_score"
        ]
    ]
    .sort_values("mean_test_score", ascending=False)
    .head(15)
    .reset_index(drop=True)
)

top_results.columns = [
    "n_features",
    "C",
    "penalty",
    "mean_cv_auc",
    "sd_cv_auc",
    "mean_train_auc"
]

display(top_results.round(4))

,n_features,C,penalty,mean_cv_auc,sd_cv_auc,mean_train_auc
0,34,0.10,l1,0.8702,0.0157,0.8796
1,30,0.10,l1,0.8701,0.0158,0.8796
2,25,0.10,l1,0.8695,0.0159,0.8794
3,34,0.01,l2,0.8693,0.0156,0.8777
4,20,0.10,l1,0.8690,0.0154,0.8791
5,25,0.10,l2,0.8689,0.0179,0.8830
6,34,0.10,l2,0.8687,0.0179,0.8833
7,30,0.10,l2,0.8686,0.0179,0.8831
8,20,0.10,l2,0.8684,0.0160,0.8824
9,30,0.01,l2,0.8682,0.0165,0.8779


In [15]:
# ============================================================
# RFE FEATURE SELECTION RESULT
# ============================================================

best_pipeline = grid_search.best_estimator_
rfe_model = best_pipeline.named_steps["rfe"]

selected_mask = rfe_model.support_

feature_selection = pd.DataFrame({
    "feature": predictors,
    "selected": selected_mask,
    "rfe_rank": rfe_model.ranking_
})

print("Features selected:", selected_mask.sum())
print("Features available:", len(predictors))

display(feature_selection)

Features selected: 34
Features available: 34


,feature,selected,rfe_rank
0,Sex,True,1
1,Age,True,1
2,CA,True,1
3,CK,True,1
4,CREA,True,1
5,ALP,True,1
6,GGT,True,1
7,GLU,True,1
8,AST,True,1
9,ALT,True,1


In [16]:
# ============================================================
# LOGISTIC REGRESSION COEFFICIENTS
# ============================================================

classifier = best_pipeline.named_steps["classifier"]

selected_features = np.array(predictors)[selected_mask]

coefficients = pd.DataFrame({
    "feature": selected_features,
    "coefficient": classifier.coef_[0]
})

coefficients["abs_coefficient"] = coefficients["coefficient"].abs()

coefficients = coefficients.sort_values(
    "abs_coefficient",
    ascending=False
).reset_index(drop=True)

display(coefficients.round(4))

,feature,coefficient,abs_coefficient
0,WBC,-0.7452,0.7452
1,CA,-0.6805,0.6805
2,LDH,0.4403,0.4403
3,Suspect,0.4368,0.4368
4,PCR,0.3559,0.3559
5,HCT,0.2866,0.2866
6,EO,-0.2845,0.2845
7,BA,-0.2018,0.2018
8,RBC,0.1830,0.1830
9,ALT,0.1231,0.1231


In [17]:
# ============================================================
# HOLD-OUT TEST SET EVALUATION
# ============================================================

# Predicted class probabilities
y_prob = best_pipeline.predict_proba(X_test)[:, 1]

# Predicted classes using default 0.50 threshold
y_pred = best_pipeline.predict(X_test)

# Confusion matrix
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

# Performance metrics
accuracy = accuracy_score(y_test, y_pred)
sensitivity = recall_score(y_test, y_pred)
specificity = tn / (tn + fp)
auc = roc_auc_score(y_test, y_prob)
brier = brier_score_loss(y_test, y_prob)

print("HOLD-OUT TEST PERFORMANCE")
print("=" * 40)

print(f"ROC-AUC:     {auc:.4f}")
print(f"Accuracy:    {accuracy:.4f}")
print(f"Sensitivity: {sensitivity:.4f}")
print(f"Specificity: {specificity:.4f}")
print(f"Brier score: {brier:.4f}")

print("\nCONFUSION MATRIX")
print("=" * 40)
print(f"True negatives:  {tn}")
print(f"False positives: {fp}")
print(f"False negatives: {fn}")
print(f"True positives:  {tp}")

HOLD-OUT TEST PERFORMANCE
ROC-AUC:     0.8770
Accuracy:    0.8000
Sensitivity: 0.8089
Specificity: 0.7917
Brier score: 0.1417

CONFUSION MATRIX
True negatives:  133
False positives: 35
False negatives: 30
True positives:  127


In [18]:
# ============================================================
# SAVE LOGISTIC REGRESSION REPRODUCTION RESULTS
# ============================================================

lr_results = pd.DataFrame({
    "model": ["Logistic Regression"],
    "cv_auc": [grid_search.best_score_],
    "holdout_auc": [auc],
    "accuracy": [accuracy],
    "sensitivity": [sensitivity],
    "specificity": [specificity],
    "brier_score": [brier],
    "true_negative": [tn],
    "false_positive": [fp],
    "false_negative": [fn],
    "true_positive": [tp],
    "n_train": [len(X_train)],
    "n_holdout": [len(X_test)],
    "n_features_rfe": [selected_mask.sum()],
    "n_nonzero_coefficients": [
        (coefficients["coefficient"] != 0).sum()
    ],
    "best_C": [grid_search.best_params_["classifier__C"]],
    "best_penalty": [
        grid_search.best_params_["classifier__penalty"]
    ]
})

display(lr_results.round(4))

,model,cv_auc,holdout_auc,accuracy,sensitivity,specificity,brier_score,true_negative,false_positive,false_negative,true_positive,n_train,n_holdout,n_features_rfe,n_nonzero_coefficients,best_C,best_penalty
0,Logistic Regression,0.8702,0.877,0.8,0.8089,0.7917,0.1417,133,35,30,127,1299,325,34,18,0.1,l1


In [19]:
# ============================================================
# EXPORT AGGREGATE RESULTS
# ============================================================

RESULTS_DIR = Path("../results/reproduction")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

lr_results.to_csv(
    RESULTS_DIR / "logistic_regression_holdout_results.csv",
    index=False
)

coefficients.to_csv(
    RESULTS_DIR / "logistic_regression_coefficients.csv",
    index=False
)

top_results.to_csv(
    RESULTS_DIR / "logistic_regression_cv_results_top15.csv",
    index=False
)

print("Results saved to:")
print(RESULTS_DIR.resolve())

Results saved to:
C:\Users\bl\Documents\ML-Projects\clinical-ml\results\reproduction


In [20]:
from pathlib import Path

print("Current working directory:")
print(Path.cwd())

print("\nNotebook data path resolves to:")
print(DATA_PATH.resolve())

print("\nResults directory resolves to:")
print(RESULTS_DIR.resolve())

Current working directory:
c:\Users\bl\Documents\ML-Projects\clinical-ml\notebooks

Notebook data path resolves to:
C:\Users\bl\Documents\ML-Projects\clinical-ml\data\raw\all_training.csv

Results directory resolves to:
C:\Users\bl\Documents\ML-Projects\clinical-ml\results\reproduction
